Imports

In [ ]:
import open_clip
from pathlib import Path
import torch
from collections import OrderedDict
from safetensors.torch import save_file


Supported built-in models in openclip repo

In [ ]:
# lists all built-in supported models. There is not a 1 to 1 mapping from the built-in models to the ones used in the scaling law paper, therefore load checkpoints directly.
open_clip.list_pretrained()

Download models from HF


In [ ]:
!python download_models.py --model "ViT-B-16" "ViT-B-32" --samples_seen 34B --dataset 2B

Zero-shot evaluations

In [ ]:
# ViT-B-16, 2B, 34B samples seen experiments for zero shot classification on mnist, cifar10, cifar100, imagenet1k, imagenetv2

def convert_pt_to_safetensors(pt_path: Path) -> Path:
    pt_path = Path(pt_path)
    assert pt_path.is_file(), f"Missing: {pt_path}"
    dst = pt_path.with_suffix(".safetensors")

    obj = torch.load(pt_path, map_location="cpu", weights_only=False)
    state = obj.get("state_dict", obj)
    if any(k.startswith("module.") for k in state):
        state = OrderedDict((k.replace("module.", "", 1), v) for k, v in state.items())

    tensor_only = {k: v for k, v in state.items() if torch.is_tensor(v)}
    save_file(tensor_only, str(dst))
    return dst

root = Path("openclip_checkpoints")
b16_pt = root / "Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.pt"
b32_pt = root / "Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.pt"

convert_pt_to_safetensors(b16_pt)
convert_pt_to_safetensors(b32_pt)

print("running evaluations")

!clip_benchmark eval --dataset=cifar10 --task zeroshot_classification --model ViT-B-16 --pretrained openclip_checkpoints/Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.safetensors --batch_size 128 --output cifar10_B-16_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=cifar100 --task zeroshot_classification --model ViT-B-16 --pretrained openclip_checkpoints/Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.safetensors --batch_size 128 --output cifar100_B-16_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=mnist --task zeroshot_classification --model ViT-B-16 --pretrained openclip_checkpoints/Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.safetensors --batch_size 128 --output mnist_B-16_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=imagenet1k --dataset_root ./root/imagenet1k  --task zeroshot_classification --model ViT-B-16 --pretrained openclip_checkpoints/Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.safetensors --batch_size 128 --output imagenet1k_B-16_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=imagenetv2 --task zeroshot_classification --model ViT-B-16 --pretrained openclip_checkpoints/Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.safetensors --batch_size 128 --output imagenetv2_B-16_LAION2B_34B.json --no_amp --seed 0

# ViT-B-32, 2B, 34B samples seen experiments for zero shot classification on mnist, cifar10, cifar100, imagenet1k, imagenetv2

!clip_benchmark eval --dataset=mnist --task zeroshot_classification --model ViT-B-32 --pretrained openclip_checkpoints/Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.safetensors --batch_size 128 --output mnist_B-32_LAION2B_34B.json --no_amp  --seed 0

!clip_benchmark eval --dataset=cifar10 --task zeroshot_classification --model ViT-B-32 --pretrained openclip_checkpoints/Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.safetensors --batch_size 128 --output cifar10_B-32_LAION2B_34B.json --no_amp  --seed 0

!clip_benchmark eval --dataset=cifar100 --task zeroshot_classification --model ViT-B-32 --pretrained openclip_checkpoints/Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.safetensors --batch_size 128 --output cifar100_B-32_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=imagenet1k --dataset_root ./root/imagenet1k --task zeroshot_classification --model ViT-B-32 --pretrained openclip_checkpoints/Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.safetensors --batch_size 128 --output imagenet1k_B-32_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=imagenetv2 --task zeroshot_classification --model ViT-B-32 --pretrained openclip_checkpoints/Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.safetensors --batch_size 128 --output imagenetv2_B-32_LAION2B_34B.json --no_amp --seed 0

print("finished evaluations")

Zero-shot image/text retrieval

In [ ]:
# MS-COCO dataset

!clip_benchmark eval --dataset=mscoco_captions --task=zeroshot_retrieval --model ViT-B-16 --pretrained openclip_checkpoints/Model-B-16_Data-2B_Samples-34B_lr-1e-3_bs-88k.safetensors --batch_size 128 --output mscoco_B-16_LAION2B_34B.json --no_amp --seed 0

!clip_benchmark eval --dataset=mscoco_captions --task=zeroshot_retrieval --model ViT-B-32 --pretrained openclip_checkpoints/Model-B-32_Data-2B_Samples-34B_lr-1e-3_bs-79k.safetensors --batch_size 128 --output mscoco_B-32_LAION2B_34B.json --no_amp --seed 0

Merging results

In [2]:
!clip_benchmark build ./ --output merged_results.csv